In [36]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Cloning repository
!git clone https://github.com/Haz1q1105/nlp-assignment3
%cd nlp-assignment3

# Install dependencies
!pip install sentencepiece matplotlib tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'nlp-assignment3'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 1), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), done.
Resolving deltas: 100% (1/1), done.
/content/nlp-assignment3/nlp-assignment3
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conf

In [37]:
!git config --global user.email "haziqmubi667@gmail.com"
!git config --global user.name "Haziq Mubashir"

In [38]:
!touch test.txt

In [39]:
!git add .
!git commit -m "Test commit from Colab"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [41]:
!wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Electronics_5.json.gz

--2026-04-24 10:36:24--  http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Electronics_5.json.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 495854086 (473M) [application/x-gzip]
Saving to: ‘reviews_Electronics_5.json.gz’

reviews_Electronics 100%[===================>] 472.88M  52.0MB/s    in 11s     

2026-04-24 10:36:35 (43.0 MB/s) - ‘reviews_Electronics_5.json.gz’ saved [495854086/495854086]



In [42]:
!wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Books_5.json.gz

--2026-04-24 10:36:36--  http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Books_5.json.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3223678899 (3.0G) [application/x-gzip]
Saving to: ‘reviews_Books_5.json.gz’

reviews_Books_5.jso 100%[===================>]   3.00G  54.0MB/s    in 69s     

2026-04-24 10:37:45 (44.5 MB/s) - ‘reviews_Books_5.json.gz’ saved [3223678899/3223678899]



In [43]:
!wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Clothing_Shoes_and_Jewelry_5.json.gz

--2026-04-24 10:37:45--  http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Clothing_Shoes_and_Jewelry_5.json.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47285200 (45M) [application/x-gzip]
Saving to: ‘reviews_Clothing_Shoes_and_Jewelry_5.json.gz’

reviews_Clothing_Sh 100%[===================>]  45.09M  12.5MB/s    in 4.3s    

2026-04-24 10:37:49 (10.6 MB/s) - ‘reviews_Clothing_Shoes_and_Jewelry_5.json.gz’ saved [47285200/47285200]



In [44]:
!ls -lh

total 3.6G
-rw-r--r-- 1 root root 3.0K Apr 24 10:36 i232579_NLP_Assignment3.ipynb
drwxr-xr-x 2 root root 4.0K Apr 24 10:36 models
drwxr-xr-x 2 root root 4.0K Apr 24 10:36 results
-rw-r--r-- 1 root root 3.1G Apr 26  2016 reviews_Books_5.json.gz
-rw-r--r-- 1 root root  46M Apr 26  2016 reviews_Clothing_Shoes_and_Jewelry_5.json.gz
-rw-r--r-- 1 root root 473M Apr 26  2016 reviews_Electronics_5.json.gz
-rw-r--r-- 1 root root    0 Apr 24 10:36 test.txt


In [45]:
import gzip
import json
from tqdm import tqdm

def load_category(file_path, max_samples=15000):
    data = []

    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f)):
            if i >= max_samples:
                break

            review = json.loads(line)

            # NOTE: same keys here
            if 'reviewText' in review and 'overall' in review:
                data.append((review['reviewText'], int(review['overall'])))

    return data

In [46]:
electronics = load_category("reviews_Electronics_5.json.gz")
books = load_category("reviews_Books_5.json.gz")
clothing = load_category("reviews_Clothing_Shoes_and_Jewelry_5.json.gz")

data = electronics + books + clothing

print("Total dataset size:", len(data))
print("Sample:", data[0])

15000it [00:00, 62902.17it/s]
15000it [00:00, 54107.11it/s]
15000it [00:00, 77303.43it/s]

Total dataset size: 45000
Sample: ('We got this GPS for my husband who is an (OTR) over the road trucker.  Very Impressed with the shipping time, it arrived a few days earlier than expected...  within a week of use however it started freezing up... could of just been a glitch in that unit.  Worked great when it worked!  Will work great for the normal person as well but does have the "trucker" option. (the big truck routes - tells you when a scale is coming up ect...)  Love the bigger screen, the ease of use, the ease of putting addresses into memory.  Nothing really bad to say about the unit with the exception of it freezing which is probably one in a million and that\'s just my luck.  I contacted the seller and within minutes of my email I received a email back with instructions for an exchange! VERY impressed all the way around!', 5)
